# 1.1 数据采集 (Data Collection)

数据采集是大模型训练的第一步，数据的质量和规模直接决定模型能力的上限。

本节涵盖：
- 网页爬取
- 开源数据集整合
- 合成数据生成
- 专有数据获取

## 1. 网页爬取

**目的**：获取互联网规模的通用文本数据

**基本原理**：使用爬虫框架（如Common Crawl）批量抓取网页内容，通过URL过滤、域名去重等策略控制数据源质量。产业实践中，Common Crawl是最主要的数据源，GPT、LLaMA等模型的训练数据中均包含大量Common Crawl数据。

**关键步骤**：
1. URL调度与去重：避免重复抓取同一页面
2. 页面下载与解析：提取正文内容，去除导航栏、广告等噪声
3. 语言识别：过滤非目标语言的页面
4. 内容质量初筛：基于URL模式、域名白名单等过滤低质量站点

In [ ]:
import re
from collections import defaultdict
from urllib.parse import urlparse

class SimpleWebCrawler:
    def __init__(self):
        self.visited_urls = set()
        self.domain_count = defaultdict(int)
        self.max_per_domain = 3
        self.quality_domains = {
            'wikipedia.org', 'arxiv.org', 'github.com',
            'nature.com', 'sciencedirect.com', 'stackoverflow.com'
        }
        self.blocked_patterns = [r'/ads/', r'/popup/', r'/login', r'/privacy']

    def should_crawl(self, url):
        parsed = urlparse(url)
        domain = parsed.netloc

        if url in self.visited_urls:
            return False, 'already_visited'
        if self.domain_count[domain] >= self.max_per_domain:
            return False, 'domain_limit'
        for pattern in self.blocked_patterns:
            if re.search(pattern, parsed.path) or re.search(pattern, parsed.query):
                return False, 'blocked_pattern'
        return True, 'ok'

    def extract_text(self, html_content):
        text = re.sub(r'<script[^>]*>.*?</script>', '', html_content, flags=re.DOTALL)
        text = re.sub(r'<style[^>]*>.*?</style>', '', text, flags=re.DOTALL)
        text = re.sub(r'<nav[^>]*>.*?</nav>', '', text, flags=re.DOTALL)
        text = re.sub(r'<[^>]+>', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def crawl(self, url, html_content):
        can_crawl, reason = self.should_crawl(url)
        if not can_crawl:
            return None, reason

        self.visited_urls.add(url)
        domain = urlparse(url).netloc
        self.domain_count[domain] += 1

        text = self.extract_text(html_content)
        is_quality = any(d in domain for d in self.quality_domains)
        return {'url': url, 'text': text, 'domain': domain, 'is_quality': is_quality}, 'ok'

crawler = SimpleWebCrawler()

test_pages = [
    ('https://en.wikipedia.org/wiki/Machine_learning', '<html><body><p>Machine learning is a subset of AI that enables systems to learn from data.</p></body></html>'),
    ('https://en.wikipedia.org/wiki/Machine_learning', '<html><body><p>Duplicate URL should be skipped.</p></body></html>'),
    ('https://example.com/ads/banner', '<html><body><p>Ad content should be blocked.</p></body></html>'),
    ('https://arxiv.org/abs/2301.00001', '<html><script>var x=1;</script><body><p>Attention is all you need. This paper introduces the Transformer architecture.</p></body></html>'),
    ('https://github.com/user/repo', '<html><nav>Navigation</nav><body><p>A PyTorch implementation of Transformer.</p></body></html>'),
]

print('=== Web Crawling Demo ===')
for url, html in test_pages:
    result, reason = crawler.crawl(url, html)
    if result:
        print(f'[CRAWLED] {url}')
        print(f'  Domain: {result["domain"]} | Quality: {result["is_quality"]}')
        print(f'  Text: {result["text"][:80]}...')
    else:
        print(f'[SKIPPED] {url} -> reason: {reason}')

print(f'\nTotal crawled: {len(crawler.visited_urls)} pages')
print(f'Domain distribution: {dict(crawler.domain_count)}')

## 2. 开源数据集整合

**目的**：复用已验证的高质量数据

**基本原理**：整合Wikipedia、Books、ArXiv、GitHub等已清洗的开源语料，减少重复清洗成本。产业实践中，开源数据集通常作为高质量数据的基石，与爬取数据混合使用。

**常用开源数据集**：
- **Wikipedia**：高质量百科知识，多语言
- **Books**：长文本连贯性，叙事能力
- **ArXiv**：学术论文，科学推理能力
- **GitHub Code**：编程能力
- **StackExchange**：问答能力
- **RedPajama-V2**：经过质量评分的网页数据

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class DatasetSource:
    name: str
    category: str
    language: str
    size_gb: float
    quality_score: float
    description: str

class DatasetAggregator:
    def __init__(self):
        self.sources: List[DatasetSource] = []
        self.dedup_cache = set()

    def add_source(self, source: DatasetSource):
        if source.name not in self.dedup_cache:
            self.dedup_cache.add(source.name)
            self.sources.append(source)

    def get_total_size(self):
        return sum(s.size_gb for s in self.sources)

    def get_category_distribution(self):
        dist = defaultdict(float)
        for s in self.sources:
            dist[s.category] += s.size_gb
        return dict(dist)

    def filter_by_quality(self, min_score=0.7):
        return [s for s in self.sources if s.quality_score >= min_score]

aggregator = DatasetAggregator()

sources = [
    DatasetSource('wikipedia-en', 'encyclopedia', 'en', 20.0, 0.95, 'English Wikipedia dump'),
    DatasetSource('wikipedia-zh', 'encyclopedia', 'zh', 2.5, 0.93, 'Chinese Wikipedia dump'),
    DatasetSource('books1', 'books', 'en', 35.0, 0.85, 'Book corpus'),
    DatasetSource('arxiv', 'academic', 'en', 50.0, 0.90, 'ArXiv papers with LaTeX'),
    DatasetSource('github-code', 'code', 'multi', 100.0, 0.80, 'Open source code'),
    DatasetSource('stackexchange', 'qa', 'multi', 15.0, 0.82, 'Q&A forums'),
    DatasetSource('common-crawl', 'web', 'multi', 5000.0, 0.50, 'Raw web crawl data'),
    DatasetSource('redpajama-v2', 'web', 'en', 800.0, 0.65, 'Quality-scored web data'),
]

for src in sources:
    aggregator.add_source(src)

print('=== Open Source Dataset Aggregation ===')
print(f'Total sources: {len(aggregator.sources)}')
print(f'Total size: {aggregator.get_total_size():.1f} GB')

print('\n--- Category Distribution ---')
for cat, size in sorted(aggregator.get_category_distribution().items(), key=lambda x: -x[1]):
    print(f'  {cat:15s}: {size:8.1f} GB ({size/aggregator.get_total_size()*100:.1f}%)')

high_quality = aggregator.filter_by_quality(min_score=0.8)
hq_size = sum(s.size_gb for s in high_quality)
print(f'\nHigh quality sources (score>=0.8): {len(high_quality)}/{len(aggregator.sources)}')
print(f'High quality data size: {hq_size:.1f} GB ({hq_size/aggregator.get_total_size()*100:.1f}%)')

## 3. 合成数据生成

**目的**：补充稀缺领域或特定格式的数据

**基本原理**：利用已有强模型（如GPT-4）根据精心设计的提示生成特定领域、特定格式的训练数据，用于弥补真实数据的不足。合成数据在以下场景特别有价值：
- 医疗、法律等隐私敏感领域
- 特定格式的指令数据（SFT数据）
- 特定难度的推理数据
- 对齐训练的偏好数据

**关键挑战**：
- 多样性：避免生成过于模板化的数据
- 质量控制：过滤低质量合成数据
- 分布偏差：合成数据可能与真实分布不同

In [ ]:
import random
import json

class SyntheticDataGenerator:
    def __init__(self, seed=42):
        random.seed(seed)
        self.templates = {
            'math': [
                'If a train travels at {speed} km/h for {time} hours, how far does it go?',
                'A store has {apples} apples and sells {sold}. How many are left?',
                'What is {a} + {b} * {c}?',
            ],
            'code': [
                'Write a Python function that {task}.',
                'Implement a {algo_type} algorithm to {task}.',
                'Debug the following code that should {task}: {code_snippet}',
            ],
            'reasoning': [
                'Given that {premise}, can we conclude that {hypothesis}?',
                'If {condition1} and {condition2}, then what can we infer about {target}?',
            ],
        }
        self.task_descriptions = {
            'sorts a list of numbers', 'finds the maximum element',
            'reverses a string', 'checks if a number is prime',
        }
        self.algo_types = ['sorting', 'searching', 'dynamic programming', 'greedy']

    def generate_math_data(self, n=5):
        data = []
        for _ in range(n):
            template = random.choice(self.templates['math'])
            prompt = template.format(
                speed=random.randint(30, 200),
                time=random.randint(1, 10),
                apples=random.randint(50, 500),
                sold=random.randint(10, 100),
                a=random.randint(10, 100),
                b=random.randint(2, 20),
                c=random.randint(2, 10),
            )
            data.append({'instruction': prompt, 'category': 'math'})
        return data

    def generate_code_data(self, n=5):
        data = []
        for _ in range(n):
            template = random.choice(self.templates['code'])
            prompt = template.format(
                task=random.choice(list(self.task_descriptions)),
                algo_type=random.choice(self.algo_types),
                code_snippet='def func(x): return x',
            )
            data.append({'instruction': prompt, 'category': 'code'})
        return data

    def validate_data(self, data):
        valid = []
        for item in data:
            instr = item['instruction']
            if len(instr) < 10:
                continue
            if instr.count('{') > 0:
                continue
            valid.append(item)
        return valid

generator = SyntheticDataGenerator()

math_data = generator.generate_math_data(5)
code_data = generator.generate_code_data(5)
all_data = math_data + code_data
valid_data = generator.validate_data(all_data)

print('=== Synthetic Data Generation ===')
print(f'Generated: {len(all_data)} samples ({len(math_data)} math + {len(code_data)} code)')
print(f'Valid after filtering: {len(valid_data)}')

print('\n--- Sample Math Instructions ---')
for item in math_data[:3]:
    print(f'  [{item["category"]}] {item["instruction"]}')

print('\n--- Sample Code Instructions ---')
for item in code_data[:3]:
    print(f'  [{item["category"]}] {item["instruction"]}')

category_dist = defaultdict(int)
for item in valid_data:
    category_dist[item['category']] += 1
print(f'\nCategory distribution: {dict(category_dist)}')

## 4. 专有数据获取

**目的**：获取行业垂直领域数据

**基本原理**：通过合作、采购或内部积累获取医疗、法律、金融等垂直领域的高质量专业数据。专有数据是构建领域大模型的竞争壁垒，通常具有以下特点：
- 数据质量高，经过专业审核
- 包含领域专业术语和知识
- 数据稀缺，不易公开获取
- 可能涉及隐私和合规要求

**数据获取策略**：
- **内部积累**：企业自有业务数据（如客服对话、法律文书）
- **数据采购**：从专业数据供应商购买
- **合作获取**：与机构合作获取脱敏数据
- **众包标注**：雇佣领域专家进行标注

In [ ]:
from dataclasses import dataclass
from typing import Optional
from datetime import datetime

@dataclass
class ProprietaryDataset:
    name: str
    domain: str
    size_docs: int
    quality_score: float
    compliance_status: str
    pii_risk_level: str
    source_type: str
    license_type: str

class DataAcquisitionManager:
    def __init__(self):
        self.datasets = []
        self.compliance_checks = {
            'gdpr': lambda d: d.compliance_status in ['compliant', 'exempt'],
            'pii_safe': lambda d: d.pii_risk_level in ['low', 'none'],
            'licensed': lambda d: d.license_type in ['commercial', 'research'],
        }

    def register_dataset(self, dataset: ProprietaryDataset):
        self.datasets.append(dataset)

    def check_compliance(self, dataset: ProprietaryDataset, checks: list):
        results = {}
        for check_name in checks:
            if check_name in self.compliance_checks:
                results[check_name] = self.compliance_checks[check_name](dataset)
        return results

    def get_usable_datasets(self, domain: Optional[str] = None):
        usable = []
        for ds in self.datasets:
            if domain and ds.domain != domain:
                continue
            checks = self.check_compliance(ds, ['gdpr', 'pii_safe', 'licensed'])
            if all(checks.values()):
                usable.append(ds)
        return usable

manager = DataAcquisitionManager()

proprietary_sources = [
    ProprietaryDataset('medical-records-v1', 'medical', 500000, 0.92, 'compliant', 'low', 'partnership', 'research'),
    ProprietaryDataset('legal-cases-cn', 'legal', 200000, 0.88, 'compliant', 'medium', 'purchased', 'commercial'),
    ProprietaryDataset('financial-reports', 'finance', 100000, 0.85, 'exempt', 'low', 'internal', 'commercial'),
    ProprietaryDataset('customer-support-chat', 'general', 1000000, 0.70, 'non-compliant', 'high', 'internal', 'internal'),
    ProprietaryDataset('clinical-trials', 'medical', 80000, 0.95, 'compliant', 'none', 'partnership', 'research'),
]

for ds in proprietary_sources:
    manager.register_dataset(ds)

print('=== Proprietary Data Acquisition ===')
print(f'Registered datasets: {len(manager.datasets)}')

print('\n--- Compliance Check Results ---')
for ds in manager.datasets:
    checks = manager.check_compliance(ds, ['gdpr', 'pii_safe', 'licensed'])
    status = 'PASS' if all(checks.values()) else 'FAIL'
    print(f'  [{status}] {ds.name}: {checks}')

usable = manager.get_usable_datasets()
print(f'\nUsable datasets (all checks pass): {len(usable)}/{len(manager.datasets)}')
total_docs = sum(ds.size_docs for ds in usable)
print(f'Total usable documents: {total_docs:,}')

medical_usable = manager.get_usable_datasets(domain='medical')
print(f'\nMedical domain usable: {len(medical_usable)} datasets')
for ds in medical_usable:
    print(f'  {ds.name}: {ds.size_docs:,} docs, quality={ds.quality_score}')